In [1]:
import sbol2
from buildcompiler import BuildCompiler
from buildcompiler.abstract_translator import extract_toplevel_definition

## Multi-Design Lvl 1 Testing:

In [2]:
design_path_list = [
    "../tests/test_files/moclo_parts_circuit.xml",
    "../tests/test_files/mocloparts116.xml",
]
design_defs = []
sbol_doc = sbol2.Document()

for design in design_path_list:
    temp_doc = sbol2.Document()
    temp_doc.read(design)

    design_defs.append(extract_toplevel_definition(temp_doc))

    # sbol_doc.read("../tests/test_files/ExampleLvl2_design.xml")

print([design_def.displayId for design_def in design_defs])

['qlSBuNBL', 'i0mwvNcgH']


In [3]:
auth = "c6f22554-f057-4b4f-a597-67041d9002e5"
collections = [
    "https://synbiohub.org/user/Gon/impl_test/impl_test_collection/1",
    "https://synbiohub.org/user/Gon/Enzyme_Implementations/Enzyme_Implementations_collection/1",
]
buildcompiler = BuildCompiler(collections, "https://synbiohub.org", auth, sbol_doc)

Indexing collection: https://synbiohub.org/user/Gon/impl_test/impl_test_collection/1
Indexing collection: https://synbiohub.org/user/Gon/Enzyme_Implementations/Enzyme_Implementations_collection/1


In [4]:
print(buildcompiler.BbsI_impl, buildcompiler.BsaI_impl, buildcompiler.T4_ligase_impl)

https://synbiohub.org/user/Gon/Enzyme_Implementations/BbsI_impl/1 https://synbiohub.org/user/Gon/Enzyme_Implementations/BsaI_impl/1 https://synbiohub.org/user/Gon/Enzyme_Implementations/T4_Ligase_impl/1


In [5]:
product_doc = sbol2.Document()

buildcompiler.assembly_lvl1(design_defs, product_doc)

Success with backbone: DVK_AE_A_E and plasmids: ['pJ23100_AB_A_B', 'pB0034_BC_B_C', 'pE0030_CD_C_D', 'pB0015_DE_D_E']
Success with backbone: DVK_AE_A_E and plasmids: ['pJ23116_AB_A_B', 'pB0034_BC_B_C', 'pE0030_CD_C_D', 'pB0015_DE_D_E']


({'https://sbolcanvas.org/qlSBuNBL/1': [Plasmid:
     Name: qlSBuNBL_composite_1_A_E
     Plasmid Definition: http://buildcompiler.org/qlSBuNBL_composite_1/1
     Strain Definitions: [None]
     Plasmid Implementations: ['http://buildcompiler.org/qlSBuNBL_composite_1_impl/1']
     Strain Implementations: [None]
     Fusion Sites: ['A', 'E']
     Antibiotic Resistance: Kanamycin],
  'https://sbolcanvas.org/i0mwvNcgH/1': [Plasmid:
     Name: i0mwvNcgH_composite_1_A_E
     Plasmid Definition: http://buildcompiler.org/i0mwvNcgH_composite_1/1
     Strain Definitions: [None]
     Plasmid Implementations: ['http://buildcompiler.org/i0mwvNcgH_composite_1_impl/1']
     Strain Implementations: [None]
     Fusion Sites: ['A', 'E']
     Antibiotic Resistance: Kanamycin]},
 <sbol2.document.Document at 0x1110c3610>)

## LVL 2

In [ ]:
lvl2_design_doc = sbol2.Document()
lvl2_design_doc.read("../tests/test_files/ExampleLvl2_design.xml")


composite_plasmids, final_doc = buildcompiler.assembly_lvl2(
    lvl2_design_doc, product_name="lvl2"
)

Gen
Gen1
Plasmid:
  Name: Gen_Gen1_plas_1_simple_A_E
  Plasmid Definition: http://buildcompiler.org/Gen_Gen1_plas_1_simple/1
  Strain Definitions: [None]
  Plasmid Implementations: ['http://buildcompiler.org/Gen_Gen1_plas_1_impl/1']
  Strain Implementations: [None]
  Fusion Sites: ['A', 'E']
  Antibiotic Resistance: Kanamycin

Plasmid:
  Name: Gen1_Gen1_plas_1_simple_E_F
  Plasmid Definition: http://buildcompiler.org/Gen1_Gen1_plas_1_simple/1
  Strain Definitions: [None]
  Plasmid Implementations: ['http://buildcompiler.org/Gen1_Gen1_plas_1_impl/1']
  Strain Implementations: [None]
  Fusion Sites: ['E', 'F']
  Antibiotic Resistance: Kanamycin

Success with backbone: DVA_AF2_A_F and plasmids: ['Gen_Gen1_plas_1_simple_A_E', 'Gen1_Gen1_plas_1_simple_E_F']
Plasmid:
  Name: DVA_AF2_A_F
  Plasmid Definition: https://synbiohub.org/user/Gon/CIDARMoCloPlasmidsKit/DVA_AF2/1
  Strain Definitions: [None]
  Plasmid Implementations: ['https://synbiohub.org/user/Gon/impl_test/DVA_AF2_impl/1']
  Strai

In [7]:
print(composite_plasmids)
final_doc.write("lvl2_assembly.xml")

[Plasmid:
  Name: lvl2_1_A_F
  Plasmid Definition: http://buildcompiler.org/lvl2_1/1
  Strain Definitions: [None]
  Plasmid Implementations: ['http://buildcompiler.org/lvl2_1_impl/1']
  Strain Implementations: [None]
  Fusion Sites: ['A', 'F']
  Antibiotic Resistance: Ampicillin
]


'Valid.'

In [8]:
# Pull chassis from sbh
chassis_doc = sbol2.Document()

buildcompiler.sbh.pull(
    "https://synbiohub.org/user/Gon/Chassis/Ecoli_DH5a/1/52b575c09496ebb3e2ef9e4c272c9e733134a874/share",
    chassis_doc,
)


chassis = chassis_doc.moduleDefinitions[0]

dummy_activity = sbol2.Activity("chassis_domestication")
dummy_activity.name = "acquisistion of chassis strain"
dummy_activity.types = "http://sbols.org/v2#build"

chassis_implementation = sbol2.Implementation(f"{chassis.name}_impl")
chassis_implementation.built = chassis.identity
chassis_implementation.wasGeneratedBy = dummy_activity

chassis_doc.add(chassis_implementation)
chassis_doc.add(dummy_activity)

# chassis_doc.write("chassis_impl.xml")

In [9]:
# bacterial_transformation(composite_plasmids, chassis_implementation, chassis, final_doc)